# SinLLaMA zero-shot offensive-language evaluation

Evaluate `polyglots/SinLlama_v01` **without fine-tuning** on the same SOLD test posts under three conditions:

- `clean`: original Sinhala script
- `light`: light Singlish/transliteration noise
- `heavy`: heavy Singlish/transliteration noise

SOLD is a binary **offensive-language** dataset, so this notebook reports `NOT=0` and `OFF=1`. This is often described informally as hate-speech detection, but the narrower claim made by the results is offensive-language detection.

SinLLaMA v01 is a continually pretrained base causal language model, not an instruction-tuned classifier. The evaluation therefore compares the conditional log-likelihood of two label completions (`NO` and `YES`) under a fixed zero-shot prompt. It does not train a classification head, update model weights, use training examples, or choose a threshold on the test set.

Recommended runtime: Kaggle/Colab with one GPU and roughly 12–16 GiB VRAM using 4-bit loading. The base Meta-Llama-3 repository is gated; accept its licence and provide `HF_TOKEN` as a secret.

## 1. Install dependencies

Run once, then restart the runtime if `bitsandbytes` was newly installed.

In [ ]:
%pip install -q -U "transformers>=4.45" "peft>=0.13" "accelerate>=1.0" "bitsandbytes>=0.44" pandas scikit-learn matplotlib seaborn tqdm

## 2. Configuration

For a quick end-to-end check, set `MAX_ROWS = 50`. Restore it to `None` for the publishable full-test result. Keep the prompt, verbalizers, and decision rule fixed across all three conditions.

In [ ]:
from pathlib import Path
import gc
import json
import os
import platform
import time

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

SEED = 42
MODEL_ID = "polyglots/SinLlama_v01"
BASE_MODEL_ID = "meta-llama/Meta-Llama-3-8B"
DATA_ROOT = None  # e.g. Path('/kaggle/input/my-dataset/data'); auto-detected when None
OUTPUT_DIR = Path("sinllama/zero_shot_results")
MAX_ROWS = None   # 50 for a smoke test; None evaluates all 2,500 paired posts
BATCH_SIZE = 4
MAX_LENGTH = 192
USE_4BIT = True
LENGTH_NORMALIZE_LABEL_SCORE = True
BOOTSTRAP_SAMPLES = 2000

# One fixed prompt for every condition. The text is delimited to reduce prompt injection.
PROMPT_TEMPLATE = """Binary offensive-language classification.
Decide whether the social-media text contains insults, threats, profanity, demeaning language, or attacks against a person or group. The text may be Sinhala script or romanized Sinhala (Singlish).
Reply YES if it is offensive; otherwise reply NO.
Text begins:
---
{text}
---
Answer:"""
LABEL_TEXT = {0: " NO", 1: " YES"}
LABEL_NAME = {0: "NOT", 1: "OFF"}

np.random.seed(SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("torch", torch.__version__, "| CUDA", torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"cuda:{i} {props.name}, {props.total_memory / 2**30:.1f} GiB")

## 3. Authenticate and locate the paired test data

No token is stored in this notebook. Set an `HF_TOKEN` environment/Kaggle secret, or run `huggingface_hub.login()` interactively before the model-loading cell.

In [ ]:
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=False)
    print("Authenticated from HF_TOKEN.")
else:
    print("HF_TOKEN is not set. Cached credentials will be used; call login() if needed.")

REQUIRED = [
    Path("sold/test.jsonl"),
    Path("outputs/test/transliteration_light.jsonl"),
    Path("outputs/test/transliteration_heavy.jsonl"),
]

def is_data_root(path):
    path = Path(path)
    return all((path / rel).is_file() for rel in REQUIRED)

def find_data_root(explicit=None):
    if explicit is not None:
        explicit = Path(explicit)
        if not is_data_root(explicit):
            raise FileNotFoundError(f"DATA_ROOT does not contain the required files: {explicit.resolve()}")
        return explicit
    cwd = Path.cwd().resolve()
    for parent in (cwd, *cwd.parents):
        for candidate in (parent / "data", parent):
            if is_data_root(candidate):
                return candidate
    kaggle = Path("/kaggle/input")
    if kaggle.exists():
        for test_file in kaggle.glob("**/sold/test.jsonl"):
            candidate = test_file.parent.parent
            if is_data_root(candidate):
                return candidate
    raise FileNotFoundError("Could not find data/. Set DATA_ROOT explicitly.")

DATA_ROOT = find_data_root(DATA_ROOT)
FILES = {
    "clean": DATA_ROOT / "sold/test.jsonl",
    "light": DATA_ROOT / "outputs/test/transliteration_light.jsonl",
    "heavy": DATA_ROOT / "outputs/test/transliteration_heavy.jsonl",
}
print("DATA_ROOT =", DATA_ROOT.resolve())
FILES

## 4. Load and verify alignment

The same post IDs and gold labels must appear in clean, light, and heavy. Alignment by ID prevents apparently plausible but invalid comparisons caused by row-order differences.

In [ ]:
def read_jsonl(path):
    with Path(path).open(encoding="utf-8") as handle:
        return pd.DataFrame(json.loads(line) for line in handle if line.strip())

def canonicalize(df, condition):
    id_col = "post_id" if condition == "clean" else "source_id"
    text_col = "text" if condition == "clean" else "noisy_text"
    missing = {id_col, text_col, "label"} - set(df.columns)
    if missing:
        raise KeyError(f"{condition}: missing columns {sorted(missing)}")
    out = df[[id_col, text_col, "label"]].rename(
        columns={id_col: "id", text_col: "text"}
    ).copy()
    out["id"] = out["id"].astype(str)
    out["text"] = out["text"].astype(str)
    out["label"] = out["label"].astype(int)
    out["condition"] = condition
    if out["id"].duplicated().any():
        raise ValueError(f"{condition}: duplicate IDs found")
    if not set(out["label"]).issubset({0, 1}):
        raise ValueError(f"{condition}: labels must be 0/1")
    if out["text"].str.strip().eq("").any():
        raise ValueError(f"{condition}: empty text found")
    return out

frames = {name: canonicalize(read_jsonl(path), name) for name, path in FILES.items()}
clean_ids = frames["clean"]["id"].tolist()
clean_id_set = set(clean_ids)
clean_labels = frames["clean"].set_index("id")["label"]
for name, frame in frames.items():
    assert set(frame["id"]) == clean_id_set, f"{name}: ID set differs from clean"
    aligned_labels = frame.set_index("id").loc[clean_ids, "label"]
    assert np.array_equal(aligned_labels.to_numpy(), clean_labels.loc[clean_ids].to_numpy()), (
        f"{name}: labels differ from clean"
    )

if MAX_ROWS is not None:
    rng = np.random.default_rng(SEED)
    selected = rng.choice(clean_ids, size=min(MAX_ROWS, len(clean_ids)), replace=False).tolist()
else:
    selected = clean_ids

for name in frames:
    frames[name] = frames[name].set_index("id").loc[selected].reset_index()
    counts = frames[name]["label"].value_counts().sort_index().to_dict()
    print(f"{name:5s}: n={len(frames[name])}, labels={counts}")

display(pd.DataFrame({
    "clean": frames["clean"]["text"].head(3),
    "light": frames["light"]["text"].head(3),
    "heavy": frames["heavy"]["text"].head(3),
    "label": frames["clean"]["label"].head(3).map(LABEL_NAME),
}))

## 5. Load SinLLaMA without task fine-tuning

SinLLaMA v01 is distributed as a PEFT causal-LM adapter over Meta-Llama-3-8B and includes extended Sinhala embeddings. The base is loaded first, resized to the SinLLaMA tokenizer, then the published adapter is attached. Nothing is trained in this notebook.

In [ ]:
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if USE_4BIT and not torch.cuda.is_available():
    raise RuntimeError("USE_4BIT=True requires a CUDA GPU. Use a GPU runtime or set USE_4BIT=False.")

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

compute_dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
)
load_kwargs = {
    "device_map": "auto",
    "low_cpu_mem_usage": True,
    "torch_dtype": compute_dtype if torch.cuda.is_available() else torch.float32,
}
if USE_4BIT:
    load_kwargs["quantization_config"] = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )

base_model = AutoModelForCausalLM.from_pretrained(BASE_MODEL_ID, **load_kwargs)
old_vocab = base_model.get_input_embeddings().weight.shape[0]
if old_vocab != len(tokenizer):
    print(f"Resizing vocabulary {old_vocab} -> {len(tokenizer)} before attaching SinLLaMA")
    base_model.resize_token_embeddings(len(tokenizer), mean_resizing=False)
model = PeftModel.from_pretrained(base_model, MODEL_ID, is_trainable=False)
model.eval()
model.config.use_cache = True
input_device = model.get_input_embeddings().weight.device

print("model training mode:", model.training)
print("tokenizer size:", len(tokenizer))
print("input device:", input_device)
for label, completion in LABEL_TEXT.items():
    ids = tokenizer.encode(completion, add_special_tokens=False)
    print(f"{LABEL_NAME[label]} completion {completion!r}: token IDs={ids}, tokens={tokenizer.convert_ids_to_tokens(ids)}")

## 6. Conditional label-likelihood classifier

For each post, compute the log-probability of the tokens in ` NO` and ` YES` after the same prompt. The larger score is the prediction. Scores are length-normalized if a verbalizer happens to tokenize into more than one token. The softmax over the two scores is reported as `p_off`; treat it as a relative model score, not a calibrated probability.

In [ ]:
def score_completion_batch(prompts, completion):
    answer_ids = tokenizer.encode(completion, add_special_tokens=False)
    if not answer_ids:
        raise ValueError(f"Completion produced no tokens: {completion!r}")
    encoded = tokenizer(
        prompts,
        add_special_tokens=True,
        truncation=True,
        max_length=MAX_LENGTH - len(answer_ids),
    )["input_ids"]
    full_ids = [prompt_ids + answer_ids for prompt_ids in encoded]
    max_len = max(map(len, full_ids))
    input_ids = torch.full(
        (len(full_ids), max_len), tokenizer.pad_token_id, dtype=torch.long
    )
    attention_mask = torch.zeros_like(input_ids)
    for row, ids in enumerate(full_ids):
        input_ids[row, :len(ids)] = torch.tensor(ids)
        attention_mask[row, :len(ids)] = 1
    input_ids = input_ids.to(input_device)
    attention_mask = attention_mask.to(input_device)

    with torch.inference_mode():
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits.float()

    scores = []
    for row, prompt_ids in enumerate(encoded):
        start = len(prompt_ids)
        positions = torch.arange(start, start + len(answer_ids), device=logits.device)
        token_logits = logits[row, positions - 1]
        targets = torch.tensor(answer_ids, device=logits.device)
        token_logp = token_logits.log_softmax(dim=-1).gather(1, targets[:, None]).squeeze(1)
        score = token_logp.mean() if LENGTH_NORMALIZE_LABEL_SCORE else token_logp.sum()
        scores.append(float(score.cpu()))
    del logits, input_ids, attention_mask
    return scores

def predict_condition(frame, condition):
    rows = []
    started = time.time()
    iterator = range(0, len(frame), BATCH_SIZE)
    for start in tqdm(iterator, desc=condition):
        batch = frame.iloc[start:start + BATCH_SIZE]
        prompts = [PROMPT_TEMPLATE.format(text=text) for text in batch["text"]]
        score_not = score_completion_batch(prompts, LABEL_TEXT[0])
        score_off = score_completion_batch(prompts, LABEL_TEXT[1])
        pair_scores = np.column_stack([score_not, score_off])
        shifted = pair_scores - pair_scores.max(axis=1, keepdims=True)
        probs = np.exp(shifted) / np.exp(shifted).sum(axis=1, keepdims=True)
        preds = pair_scores.argmax(axis=1)
        for (_, item), s0, s1, pred, p_off in zip(
            batch.iterrows(), score_not, score_off, preds, probs[:, 1]
        ):
            rows.append({
                "id": item["id"], "condition": condition,
                "text": item["text"], "label": int(item["label"]),
                "prediction": int(pred), "score_not": s0,
                "score_off": s1, "p_off": float(p_off),
            })
    result = pd.DataFrame(rows)
    result["correct"] = result["label"] == result["prediction"]
    print(f"{condition}: {len(result)} rows in {(time.time() - started) / 60:.1f} min")
    return result

prediction_frames = [predict_condition(frames[name], name) for name in ("clean", "light", "heavy")]
predictions = pd.concat(prediction_frames, ignore_index=True)
predictions.to_json(
    OUTPUT_DIR / "predictions.jsonl", orient="records", lines=True, force_ascii=False
)
predictions.head()

## 7. Metrics and paired robustness results

Macro-F1 is the primary metric because SOLD is imbalanced. The paired robustness table additionally measures prediction flips and regressions on the exact same posts. Bootstrap intervals estimate uncertainty in each noisy-minus-clean macro-F1 difference.

In [ ]:
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, classification_report,
    confusion_matrix, f1_score, precision_recall_fscore_support, roc_auc_score,
)

def metric_row(frame, condition):
    y = frame["label"].to_numpy()
    pred = frame["prediction"].to_numpy()
    precision, recall, f1, support = precision_recall_fscore_support(
        y, pred, labels=[0, 1], zero_division=0
    )
    try:
        auc = roc_auc_score(y, frame["p_off"])
    except ValueError:
        auc = np.nan
    return {
        "condition": condition,
        "n": len(frame),
        "accuracy": accuracy_score(y, pred),
        "balanced_accuracy": balanced_accuracy_score(y, pred),
        "macro_f1": f1_score(y, pred, average="macro", zero_division=0),
        "weighted_f1": f1_score(y, pred, average="weighted", zero_division=0),
        "not_precision": precision[0], "not_recall": recall[0], "not_f1": f1[0],
        "off_precision": precision[1], "off_recall": recall[1], "off_f1": f1[1],
        "roc_auc": auc,
    }

metrics = pd.DataFrame([
    metric_row(predictions[predictions["condition"] == name], name)
    for name in ("clean", "light", "heavy")
])
metrics.to_csv(OUTPUT_DIR / "metrics.csv", index=False)
display(metrics.round(4))

for name in ("clean", "light", "heavy"):
    part = predictions[predictions["condition"] == name]
    print(f"\n=== {name.upper()} ===")
    print(classification_report(
        part["label"], part["prediction"],
        labels=[0, 1], target_names=["NOT", "OFF"], digits=4, zero_division=0,
    ))

In [ ]:
def paired_bootstrap_delta(clean, noisy, samples=2000, seed=42):
    merged = clean[["id", "label", "prediction"]].merge(
        noisy[["id", "label", "prediction"]],
        on=["id", "label"], suffixes=("_clean", "_noisy"), validate="one_to_one",
    )
    y = merged["label"].to_numpy()
    pc = merged["prediction_clean"].to_numpy()
    pn = merged["prediction_noisy"].to_numpy()
    rng = np.random.default_rng(seed)
    deltas = np.empty(samples)
    for b in range(samples):
        idx = rng.integers(0, len(merged), len(merged))
        deltas[b] = (
            f1_score(y[idx], pn[idx], average="macro", zero_division=0)
            - f1_score(y[idx], pc[idx], average="macro", zero_division=0)
        )
    return np.quantile(deltas, [0.025, 0.975])

by_condition = {name: predictions[predictions["condition"] == name].set_index("id")
                for name in ("clean", "light", "heavy")}
clean = by_condition["clean"]
clean_f1 = float(metrics.set_index("condition").loc["clean", "macro_f1"])
robustness_rows = []
for name in ("light", "heavy"):
    noisy = by_condition[name].loc[clean.index]
    noisy_f1 = float(metrics.set_index("condition").loc[name, "macro_f1"])
    clean_correct = clean["correct"].to_numpy()
    noisy_correct = noisy["correct"].to_numpy()
    ci_low, ci_high = paired_bootstrap_delta(
        clean.reset_index(), noisy.reset_index(), BOOTSTRAP_SAMPLES, SEED
    )
    robustness_rows.append({
        "condition": name,
        "macro_f1": noisy_f1,
        "delta_vs_clean": noisy_f1 - clean_f1,
        "degradation_clean_minus_noisy": clean_f1 - noisy_f1,
        "delta_ci95_low": ci_low, "delta_ci95_high": ci_high,
        "prediction_flip_rate": (clean["prediction"].to_numpy() != noisy["prediction"].to_numpy()).mean(),
        "clean_correct_to_noisy_wrong_rate_all": (clean_correct & ~noisy_correct).mean(),
        "regression_rate_given_clean_correct": (clean_correct & ~noisy_correct).sum() / max(clean_correct.sum(), 1),
        "noisy_correct_from_clean_wrong_rate_all": (~clean_correct & noisy_correct).mean(),
    })
robustness = pd.DataFrame(robustness_rows)
robustness.to_csv(OUTPUT_DIR / "robustness.csv", index=False)
display(robustness.round(4))

## 8. Visualize and save a reproducible summary

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 4, figsize=(18, 4))
sns.barplot(data=metrics, x="condition", y="macro_f1", ax=axes[0], color="#4C72B0")
axes[0].set_ylim(0, 1)
axes[0].set_title("Macro-F1 (primary)")
for ax, name in zip(axes[1:], ("clean", "light", "heavy")):
    part = predictions[predictions["condition"] == name]
    cm = confusion_matrix(part["label"], part["prediction"], labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["NOT", "OFF"], yticklabels=["NOT", "OFF"])
    ax.set_title(name.capitalize())
    ax.set_xlabel("Predicted")
    ax.set_ylabel("Gold")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "evaluation.png", dpi=180, bbox_inches="tight")
plt.show()

summary = {
    "method": "zero-shot conditional label likelihood; no task fine-tuning",
    "model_id": MODEL_ID, "base_model_id": BASE_MODEL_ID,
    "seed": SEED, "max_rows": MAX_ROWS, "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE, "use_4bit": USE_4BIT,
    "prompt_template": PROMPT_TEMPLATE, "label_text": LABEL_TEXT,
    "length_normalize_label_score": LENGTH_NORMALIZE_LABEL_SCORE,
    "bootstrap_samples": BOOTSTRAP_SAMPLES,
    "metrics": metrics.to_dict(orient="records"),
    "robustness": robustness.to_dict(orient="records"),
    "environment": {
        "python": platform.python_version(), "torch": torch.__version__,
        "cuda_available": torch.cuda.is_available(),
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    },
}
with (OUTPUT_DIR / "summary.json").open("w", encoding="utf-8") as handle:
    json.dump(summary, handle, ensure_ascii=False, indent=2)
print("Saved outputs to", OUTPUT_DIR.resolve())
print(sorted(path.name for path in OUTPUT_DIR.iterdir()))

## 9. Inspect paired errors

These are cases that the clean input gets right but a noisy version gets wrong. Use them for qualitative analysis only; do not edit the prompt after inspecting test errors and then report the same test-set score as confirmatory.

In [ ]:
wide = predictions.pivot(index=["id", "label"], columns="condition", values=["text", "prediction", "p_off"])
wide.columns = [f"{field}_{condition}" for field, condition in wide.columns]
wide = wide.reset_index()
for name in ("light", "heavy"):
    mask = (wide["prediction_clean"] == wide["label"]) & (wide[f"prediction_{name}"] != wide["label"])
    print(f"\nClean-correct -> {name}-wrong examples: {mask.sum()}")
    display(wide.loc[mask, [
        "id", "label", "text_clean", f"text_{name}",
        "prediction_clean", f"prediction_{name}", "p_off_clean", f"p_off_{name}",
    ]].head(20))

## Reporting checklist

Report the model revision/date, exact prompt and verbalizers, quantization, truncation length, test size, class mapping, all three macro-F1 values, OFF precision/recall/F1, paired degradation and confidence intervals. State explicitly that `p_off` is not calibrated and that the model card describes SinLLaMA v01 as a non-instruction-tuned base model requiring task-specific fine-tuning for robust downstream use. A low zero-shot score is therefore a valid baseline result, not evidence that the notebook failed.